# 00 · Prepare VisDrone once

Run all cells. The notebook is idempotent: verified archives, extraction,
conversion, and LR-search manifests are reused. Raw VisDrone data is never
modified. The controlled workflow prepares only the 2-class track by default.


In [ ]:
USE_GOOGLE_DRIVE = True
DATASET_SOURCE = "auto"
PREPARE_10CLASS_TRACK = False
REDOWNLOAD = False
SMOKE_TEST = False


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
IN_KAGGLE = not IN_COLAB and bool(
    os.environ.get("KAGGLE_KERNEL_RUN_TYPE")
    or os.environ.get("KAGGLE_URL_BASE")
    or Path("/kaggle/working").is_dir()
)
NOTEBOOK_PLATFORM = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"
SMOKE_TEST = SMOKE_TEST or os.environ.get("SMOKE_TEST", "").lower() in {"1", "true", "yes"}

repository_default = (
    Path("/content/aerial-object-detection-benchmark")
    if IN_COLAB
    else Path("/kaggle/working/aerial-object-detection-benchmark")
    if IN_KAGGLE
    else Path.cwd()
)
repository_override = os.environ.get("BENCHMARK_REPO_ROOT")
repository_candidates = (
    [Path(repository_override).expanduser()]
    if repository_override
    else [Path.cwd(), *Path.cwd().parents, repository_default]
)
REPO_PATH = next(
    (
        candidate.resolve()
        for candidate in repository_candidates
        if (candidate / "pyproject.toml").is_file()
        and (candidate / "src" / "__init__.py").is_file()
    ),
    repository_default.resolve(),
)
git_probe = subprocess.run(
    ["git", "-C", str(REPO_PATH), "rev-parse", "--is-inside-work-tree"],
    check=False, capture_output=True, text=True,
)
if git_probe.returncode != 0:
    if NOTEBOOK_PLATFORM == "local":
        raise RuntimeError(
            "Run this notebook from the repository or set BENCHMARK_REPO_ROOT."
        )
    REPO_PATH.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--branch", "main", "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git", str(REPO_PATH)],
        check=True,
    )
else:
    current_commit = subprocess.check_output(
        ["git", "-C", str(REPO_PATH), "rev-parse", "HEAD"], text=True
    ).strip()
    print(f"Using selected repository commit {current_commit}.")
sys.path.insert(0, str(REPO_PATH))

from src.notebook_environment import setup_notebook_environment
notebook_environment = setup_notebook_environment(
    REPO_PATH,
    platform=NOTEBOOK_PLATFORM,
    use_google_drive=USE_GOOGLE_DRIVE,
    requirements_file='requirements-dataset-colab.txt',
    smoke_test=SMOKE_TEST,
)
REPO_PATH = notebook_environment.repository_root
DRIVE_ROOT = notebook_environment.artifact_root
LOCAL_CACHE_ROOT = notebook_environment.local_cache_root
print(notebook_environment.as_dict())


In [ ]:
if SMOKE_TEST:
    from src.data.download import VISDRONE_ARCHIVES
    from src.data.smoke_dataset import create_smoke_archives
    archive_dir = DRIVE_ROOT / "datasets" / "VisDrone2019-DET" / "archives"
    create_smoke_archives(archive_dir)
    smoke_manifest_dir = DRIVE_ROOT / "datasets" / "VisDrone2019-DET" / "manifests"
    for split in VISDRONE_ARCHIVES:
        (smoke_manifest_dir / f"{split}_archive.json").unlink(missing_ok=True)
        (smoke_manifest_dir / f"{split}_extraction.json").unlink(missing_ok=True)
    for spec in VISDRONE_ARCHIVES.values():
        spec["minimum_bytes"] = 1

from src.workflows.dataset_setup import prepare_visdrone
summary = prepare_visdrone(
    REPO_PATH,
    DRIVE_ROOT,
    dataset_source=DATASET_SOURCE,
    prepare_10class_track=PREPARE_10CLASS_TRACK,
    redownload=REDOWNLOAD,
    smoke_test=SMOKE_TEST,
)


In [ ]:
import json
import random
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from src.data.dataset import CocoDetectionRecords, detection_collate

coco_root = Path(summary["coco_2class"])
records = CocoDetectionRecords(Path(summary["train_images"]), coco_root / "annotations" / "instances_train.json")
indices = random.Random(42).sample(range(len(records)), min(3, len(records)))
figure, axes = plt.subplots(1, len(indices), figsize=(5 * len(indices), 4))
axes = [axes] if len(indices) == 1 else axes
for axis, index in zip(axes, indices):
    item = records[index]
    axis.imshow(item["image"])
    for annotation in item["annotations"]:
        x, y, width, height = annotation["bbox"]
        axis.add_patch(Rectangle((x, y), width, height, fill=False, edgecolor="red", linewidth=1))
    axis.set_title(item["file_name"])
    axis.axis("off")
plt.tight_layout()
plt.show()

try:
    from torch.utils.data import DataLoader
    batch = next(iter(DataLoader(records, batch_size=2, shuffle=False, collate_fn=detection_collate)))
except ImportError:
    batch = detection_collate([records[0], records[min(1, len(records) - 1)]])
assert len(batch) == 2 and all("image" in item and "annotations" in item for item in batch)
print("DataLoader smoke test: READY")


In [ ]:
contract = summary["data_contract"]
if not contract["verified"]:
    raise RuntimeError("DATA CONTRACT VERIFIED: NO")
details = contract["details"]
print("DATA CONTRACT VERIFIED: YES")
print()
print(f"Persistent Drive root: {summary['drive_root']}")
print(f"Train archive: {details['paths']['train_archive']}")
print(f"Validation archive: {details['paths']['validation_archive']}")
print(f"Train images: {summary['train_images']}")
print(f"Validation images: {summary['validation_images']}")
print(f"Train image count: {details['extractions']['train']['recorded']['image_count']}")
print(f"Validation image count: {details['extractions']['val']['recorded']['image_count']}")
print(f"COCO train: {details['paths']['coco_train']}")
print(f"COCO validation: {details['paths']['coco_validation']}")
print(f"LR-search manifests: {summary['lr_search_manifests']}")
print("Local-cache status: NOT ENABLED IN NOTEBOOK 00")
print("Next notebook: 01_run_model_day.ipynb")
if notebook_environment.platform == "colab":
    print("https://colab.research.google.com/github/Harryphan72007/aerial-object-detection-benchmark/blob/main/notebooks/01_run_model_day.ipynb")
elif notebook_environment.platform == "kaggle":
    print("Open notebooks/01_run_model_day.ipynb in the same Kaggle session.")
